In [ ]:
# ============================================================
# TASK 3.0 — MERGE 11 YEARLY NDVI FILES INTO HISTORICAL BASELINE
# Run as a single cell in Google Colab
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import glob
import os

DRIVE_FOLDER = "/content/drive/MyDrive/Zenodo_Mekong_Data"

# --- Step 1: find and read all 11 yearly files ---
files = sorted(glob.glob(f"{DRIVE_FOLDER}/NDVI_Yearly_*.csv"))
print(f"Found {len(files)} files:")
for f in files:
    print(" -", os.path.basename(f))

expected_years = set(range(2015, 2026))
found_years = set()
dfs = []
for f in files:
    year = int(os.path.basename(f).replace("NDVI_Yearly_", "").replace(".csv", ""))
    found_years.add(year)
    df = pd.read_csv(f)
    df['year'] = year
    dfs.append(df)

missing_years = expected_years - found_years
if missing_years:
    print(f"\n⚠️ WARNING: missing {len(missing_years)} year(s): {sorted(missing_years)} — check the GEE Tasks tab before using this result.")
else:
    print(f"\n✓ All 11/11 years present (2015-2025).")

combined = pd.concat(dfs, ignore_index=True)
print(f"\nTotal {len(combined)} rows, {combined.point_id.nunique()} unique points.")

# --- Step 2: drop years with no clean scene at that point (NDVI_count == 0 or NaN) ---
valid = combined[(combined['NDVI_count'] > 0) & (combined['NDVI_mean'].notna())].copy()
print(f"After dropping empty years: {len(valid)} rows ({len(combined)-len(valid)} rows dropped).")

# --- Step 3: aggregate by point_id into historical statistics ---
stats = valid.groupby('point_id').agg(
    strata=('strata', 'first'),
    NDVI_Historical_Mean=('NDVI_mean', 'mean'),
    NDVI_Historical_StdDev=('NDVI_mean', 'std'),
    Total_Valid_Years=('NDVI_mean', 'count'),
    Total_Scenes_Used=('NDVI_count', 'sum')
).reset_index()

# Points with zero valid years drop out of the table above entirely —
# add them back with NaN so no point_id is lost before the Task 13.0 join
all_point_ids = combined[['point_id', 'strata']].drop_duplicates()
stats_full = all_point_ids.merge(stats.drop(columns='strata'), on='point_id', how='left')

n_orphan = stats_full['Total_Valid_Years'].isna().sum()
print(f"\nPoints with zero valid years (0/11): {n_orphan} / {len(stats_full)}")

# --- Step 4: quality check — flag points with few observations (low robustness) ---
print(f"\nDistribution of Total_Valid_Years:")
print(stats_full['Total_Valid_Years'].describe())
n_low = (stats_full['Total_Valid_Years'] < 3).sum()
print(f"\nPoints with <3/11 valid years (weak baseline, consider excluding from training): {n_low} ({n_low/len(stats_full):.1%})")

# --- Step 5: ecological sanity check — NDVI should rank Trees highest, Water lowest ---
print(f"\nNDVI_Historical_Mean by strata (sanity check):")
print(stats_full.groupby('strata')['NDVI_Historical_Mean'].mean().sort_values(ascending=False))

# --- Step 6: save output ---
out_path = f"{DRIVE_FOLDER}/NDVI_Historical_Stats_Final.csv"
stats_full.to_csv(out_path, index=False)
print(f"\n✓ Saved: {out_path}")
print(f"✓ {len(stats_full)} points, columns: {stats_full.columns.tolist()}")

Mounted at /content/drive
Found 11 files:
 - NDVI_Yearly_2015.csv
 - NDVI_Yearly_2016.csv
 - NDVI_Yearly_2017.csv
 - NDVI_Yearly_2018.csv
 - NDVI_Yearly_2019.csv
 - NDVI_Yearly_2020.csv
 - NDVI_Yearly_2021.csv
 - NDVI_Yearly_2022.csv
 - NDVI_Yearly_2023.csv
 - NDVI_Yearly_2024.csv
 - NDVI_Yearly_2025.csv

✓ All 11/11 years present (2015-2025).

Total 132000 rows, 12000 unique points.
After dropping empty years: 95360 rows (36640 rows dropped).

Points with zero valid years (0/11): 0 / 12000

Distribution of Total_Valid_Years:
count    12000.000000
mean         7.946667
std          0.906214
min          7.000000
25%          7.000000
50%          8.000000
75%          9.000000
max         11.000000
Name: Total_Valid_Years, dtype: float64

Points with <3/11 valid years (weak baseline, consider excluding from training): 0 (0.0%)

NDVI_Historical_Mean by strata (sanity check):
strata
1    0.791830
5    0.642129
2    0.609275
6    0.596932
4    0.593282
3    0.354617
7    0.312265
0    0.1

In [ ]:
# ============================================================
# TASK 4.0 — QA & CLEANING OF NDVI HISTORICAL BASELINE
# Run as a single cell in Google Colab (run after Task 3.0)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

DRIVE_FOLDER = "/content/drive/MyDrive/Zenodo_Mekong_Data"

# --- Step 1: read Task 3.0 output ---
df = pd.read_csv(f"{DRIVE_FOLDER}/NDVI_Historical_Stats_Final.csv")
print(f"Read {len(df)} points from Task 3.0.")

# --- Step 2: check physically valid range (NDVI must be in [-1,1]) ---
n_out_of_range = ((df['NDVI_Historical_Mean'] < -1) | (df['NDVI_Historical_Mean'] > 1)).sum()
print(f"\nNDVI values outside [-1,1] (physically invalid, must drop): {n_out_of_range}")
df = df[(df['NDVI_Historical_Mean'] >= -1) & (df['NDVI_Historical_Mean'] <= 1) | df['NDVI_Historical_Mean'].isna()]

# --- Step 3: outlier detection via IQR, computed WITHIN each stratum ---
# (not pooled across land-cover classes, since a sensible NDVI range for Water
#  differs entirely from Trees)
def flag_outliers_iqr(group, col, k=1.5):
    q1 = group[col].quantile(0.25)
    q3 = group[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return (group[col] < lower) | (group[col] > upper)

df['is_outlier_mean'] = df.groupby('strata', group_keys=False).apply(
    lambda g: flag_outliers_iqr(g, 'NDVI_Historical_Mean')
)
df['is_outlier_stddev'] = df.groupby('strata', group_keys=False).apply(
    lambda g: flag_outliers_iqr(g, 'NDVI_Historical_StdDev')
)

n_outlier_mean = df['is_outlier_mean'].sum()
n_outlier_stddev = df['is_outlier_stddev'].sum()
print(f"\nOutliers in NDVI_Historical_Mean (per-stratum IQR): {n_outlier_mean} ({n_outlier_mean/len(df):.1%})")
print(f"Outliers in NDVI_Historical_StdDev (per-stratum IQR): {n_outlier_stddev} ({n_outlier_stddev/len(df):.1%})")

print(f"\nOutlier breakdown by stratum:")
print(df.groupby('strata')['is_outlier_mean'].sum())

# --- Step 4: check weak-baseline points (already computed in Task 3.0) ---
n_weak = (df['Total_Valid_Years'] < 3).sum()
print(f"\nWeak-baseline points (<3/11 valid years): {n_weak} ({n_weak/len(df):.1%})")

# --- Step 5: composite flag — does NOT auto-delete, only flags for later decision ---
df['qa_flag'] = 'OK'
df.loc[df['is_outlier_mean'] | df['is_outlier_stddev'], 'qa_flag'] = 'OUTLIER'
df.loc[df['Total_Valid_Years'] < 3, 'qa_flag'] = 'WEAK_BASELINE'
df.loc[df['Total_Valid_Years'].isna(), 'qa_flag'] = 'NO_DATA'

print(f"\nqa_flag distribution:")
print(df['qa_flag'].value_counts())
print(f"\nRetention rate if keeping only qa_flag='OK': {(df['qa_flag']=='OK').sum()/len(df):.1%}")

# --- Step 6: save the QA-flagged version (data is NOT deleted; Task 13/14 decide later) ---
out_path = f"{DRIVE_FOLDER}/NDVI_Historical_Stats_Clean.csv"
df.to_csv(out_path, index=False)
print(f"\n✓ Saved: {out_path}")
print(f"✓ The qa_flag column lets Task 13.0/14.0 decide which points to exclude before labeling, without hard-deleting here.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Read 12000 points from Task 3.0.

NDVI values outside [-1,1] (physically invalid, must drop): 0

Outliers in NDVI_Historical_Mean (per-stratum IQR): 295 (2.5%)
Outliers in NDVI_Historical_StdDev (per-stratum IQR): 165 (1.4%)

Outlier breakdown by stratum:
strata
0      0
1    130
2     10
3      0
4     83
5     30
6     42
7      0
Name: is_outlier_mean, dtype: int64

Weak-baseline points (<3/11 valid years): 0 (0.0%)

qa_flag distribution:
qa_flag
OK         11583
OUTLIER      417
Name: count, dtype: int64

Retention rate if keeping only qa_flag='OK': 96.5%

✓ Saved: /content/drive/MyDrive/Zenodo_Mekong_Data/NDVI_Historical_Stats_Clean.csv
✓ The qa_flag column lets Task 13.0/14.0 decide which points to exclude before labeling, without hard-deleting here.


/tmp/ipykernel_726/2475901364.py:33: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df['is_outlier_mean'] = df.groupby('strata', group_keys=False).apply(
/tmp/ipykernel_726/2475901364.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df['is_outlier_stddev'] = df.groupby('strata', group_keys=False).apply(


In [ ]:
# =========================================================================
# TASK 8.0: INDEX OF INUNDATION FREQUENCY (IFI) & ANOMALY CALCULATION
# Downstream use: hydrological variable & label condition (Task 14.0)
# =========================================================================
import os
import pandas as pd
import numpy as np
from google.colab import drive

# 1. Mount Google Drive
print("Connecting to Google Drive...")
drive.mount('/content/drive')

# 2. Define paths
DRIVE_DIR = '/content/drive/MyDrive/Zenodo_Mekong_Data'

# Support both possible output filenames from Task 7.0
file_annual = os.path.join(DRIVE_DIR, 'Inundation_Otsu_Points_Annual.csv')
file_default = os.path.join(DRIVE_DIR, 'Inundation_Otsu_Points.csv')

if os.path.exists(file_annual):
    INPUT_FILE = file_annual
elif os.path.exists(file_default):
    INPUT_FILE = file_default
else:
    raise FileNotFoundError(f"Input file not found in {DRIVE_DIR}!")

OUTPUT_FILE = os.path.join(DRIVE_DIR, 'IFI_Anomaly_Points.csv')
print(f"Input file found: {INPUT_FILE}")

# 3. Read data
df_otsu = pd.read_csv(INPUT_FILE)
print(f"Data shape: {df_otsu.shape}")

# 4. Auto-detect column layout & compute IFI
inundation_cols = [c for c in df_otsu.columns if c.lower().startswith('inundation_') and c != 'Inundation_binary']

if inundation_cols:
    # Case 1: wide format (inundation_2015, inundation_2016, ...)
    print(f"Detected {len(inundation_cols)} yearly inundation columns: {inundation_cols}")
    df_otsu['Inundation_Sum'] = df_otsu[inundation_cols].sum(axis=1)
    df_otsu['N_Total'] = len(inundation_cols)
    df_otsu['IFI'] = df_otsu['Inundation_Sum'] / df_otsu['N_Total']

elif 'Inundation_binary' in df_otsu.columns and 'point_id' in df_otsu.columns:
    # Case 2: long format (1 row per timestamp per point_id)
    print("Detected long format with 'Inundation_binary' column...")
    df_otsu = df_otsu.groupby('point_id').agg(
        Inundation_Sum=('Inundation_binary', 'sum'),
        N_Total=('Inundation_binary', 'count'),
        DEM_Elevation=('DEM_Elevation', 'mean') if 'DEM_Elevation' in df_otsu.columns else ('point_id', 'count')
    ).reset_index()
    df_otsu['IFI'] = df_otsu['Inundation_Sum'] / df_otsu['N_Total']

elif 'IFI' in df_otsu.columns:
    print("'IFI' column already present in the data.")
else:
    raise KeyError("Could not find 'inundation_YYYY', 'Inundation_binary', or 'IFI' columns in the CSV!")

# 5. Compute historical P90 threshold & assign anomaly label
p90_threshold = df_otsu['IFI'].quantile(0.90)
print(f"\nRegion-wide P90 IFI threshold: {p90_threshold:.4f}")

df_otsu['IFI_P90_Threshold'] = p90_threshold
df_otsu['IFI_Anomaly'] = (df_otsu['IFI'] > p90_threshold).astype(int)

# 6. Check the resulting distribution
anomaly_count = df_otsu['IFI_Anomaly'].value_counts()
print("\nIFI Anomaly label distribution:")
print(f" - Total sample points: {len(df_otsu)}")
print(f" - Normal points (IFI <= P90): {anomaly_count.get(0, 0)}")
print(f" - Anomalous points (IFI > P90): {anomaly_count.get(1, 0)} ({anomaly_count.get(1, 0)/len(df_otsu)*100:.2f}%)")

# 7. Select standardized output columns
cols_to_keep = ['point_id', 'IFI', 'IFI_P90_Threshold', 'IFI_Anomaly']
if 'DEM_Elevation' in df_otsu.columns:
    cols_to_keep.append('DEM_Elevation')
if 'strata' in df_otsu.columns:
    cols_to_keep.append('strata')

output_df = df_otsu[cols_to_keep]

# 8. Export result
output_df.to_csv(OUTPUT_FILE, index=False)
print(f"\n✓ SAVED SUCCESSFULLY: {OUTPUT_FILE}")

Connecting to Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Input file found: /content/drive/MyDrive/Zenodo_Mekong_Data/Inundation_Otsu_Points_Annual.csv
Data shape: (12000, 25)
Detected 11 yearly inundation columns: ['inundation_2015', 'inundation_2016', 'inundation_2017', 'inundation_2018', 'inundation_2019', 'inundation_2020', 'inundation_2021', 'inundation_2022', 'inundation_2023', 'inundation_2024', 'inundation_2025']

Region-wide P90 IFI threshold: 1.0000

IFI Anomaly label distribution:
 - Total sample points: 12000
 - Normal points (IFI <= P90): 12000
 - Anomalous points (IFI > P90): 0 (0.00%)

✓ SAVED SUCCESSFULLY: /content/drive/MyDrive/Zenodo_Mekong_Data/IFI_Anomaly_Points.csv


In [ ]:
# =============================================================================
# TASK 13.0 — HARMONIZE ALL POINT-LEVEL FILES INTO ONE MASTER TABLE
# =============================================================================
import os
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = "/content/drive/MyDrive/Zenodo_Mekong_Data"

if not os.path.exists(DATA_DIR):
    print(f"⚠️ Path not found: '{DATA_DIR}'")
    print("Check your Google Drive folder name.")
else:
    print(f"✓ Connected to Drive folder: '{DATA_DIR}'")

STATIC_FILES = [
    ("NDVI_Historical_Stats", "NDVI_Historical_Stats_Final.csv"),
    ("S1_Filtered",           "S1_Filtered_Points.csv"),
    ("Inundation_Otsu",       "Inundation_Otsu_Points_Annual.csv"),
    ("IFI_Anomaly",           "IFI_Anomaly_Points_Annual.csv"),
    ("LUCR",                  "LUCR_Points_Annual.csv"),
    ("Population",            "Population_Points_Annual.csv"),
    ("Climate",               "Climate_Points_Annual.csv"),
    ("Distance",              "Distance_Points_Annual.csv"),
]

YEARLY_NDVI_FILES = [
    (f"NDVI_Yearly_{year}", f"NDVI_Yearly_{year}.csv")
    for year in range(2015, 2026)
]

ALL_FILES = STATIC_FILES + YEARLY_NDVI_FILES

df_merged = None
for idx, (label, fname) in enumerate(ALL_FILES, 1):
    fpath = os.path.join(DATA_DIR, fname)
    if not os.path.exists(fpath):
        print(f"⚠️ Warning: {fname} not found at {fpath}")
        continue

    df_step = pd.read_csv(fpath)

    if label.startswith("NDVI_Yearly_"):
        yr = label.split("_")[-1]
        df_step = df_step.rename(columns={
            "NDVI_mean": f"NDVI_mean_{yr}",
            "NDVI_count": f"NDVI_count_{yr}"
        })

    if df_merged is None:
        df_merged = df_step.copy()
        print(f"[{idx}/{len(ALL_FILES)}] Initialized anchor table from {fname}: {df_merged.shape}")
    else:
        if 'strata' in df_step.columns and 'strata' in df_merged.columns:
            df_step = df_step.drop(columns=['strata'])
        if 'DEM_Elevation' in df_step.columns and 'DEM_Elevation' in df_merged.columns:
            df_step = df_step.drop(columns=['DEM_Elevation'])

        df_merged = pd.merge(df_merged, df_step, on='point_id', how='inner')
        print(f"[{idx}/{len(ALL_FILES)}] Merged {label:22s} -> current shape: {df_merged.shape}")
        assert len(df_merged) == 12000, f"Error: rows lost while merging {fname}!"

if df_merged is not None:
    output_parquet = os.path.join(DATA_DIR, "Final_Harmonized_Points.parquet")
    output_csv = os.path.join(DATA_DIR, "Final_Harmonized_Points.csv")
    try:
        df_merged.to_parquet(output_parquet, index=False)
        print(f"\n✓ TASK 13 COMPLETE! Parquet file saved: {output_parquet}")
    except Exception as e:
        df_merged.to_csv(output_csv, index=False)
        print(f"\n✓ TASK 13 COMPLETE! CSV file saved: {output_csv}")
    print(f"Final matrix size: {df_merged.shape[0]} rows x {df_merged.shape[1]} columns")
else:
    print("❌ No data was merged. Check the Drive folder.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Connected to Drive folder: '/content/drive/MyDrive/Zenodo_Mekong_Data'
[1/19] Initialized anchor table from NDVI_Historical_Stats_Final.csv: (12000, 6)
[2/19] Merged S1_Filtered            -> current shape: (12000, 8)
[3/19] Merged Inundation_Otsu        -> current shape: (12000, 31)
[4/19] Merged IFI_Anomaly            -> current shape: (12000, 34)
[5/19] Merged LUCR                   -> current shape: (12000, 45)
[6/19] Merged Population             -> current shape: (12000, 49)
[7/19] Merged Climate                -> current shape: (12000, 51)
[8/19] Merged Distance               -> current shape: (12000, 53)
[9/19] Merged NDVI_Yearly_2015       -> current shape: (12000, 55)
[10/19] Merged NDVI_Yearly_2016       -> current shape: (12000, 57)
[11/19] Merged NDVI_Yearly_2017       -> current shape: (12000, 59)
[12/19] Merged NDVI_Yearly_2018       -> curre

In [ ]:
# ============================================================================
# TASK 1.2 (BỔ SUNG) — GẮN NHÃN RNM THẬT TỪ LỚP THAM CHIẾU CHUYÊN BIỆT (GEE)
# Chạy 1 lần trên GEE, dùng làm mặt nạ không gian cứng thay cho các bước lọc
# thống kê (NDVI floor, persistence) đã làm ở Task 20.8 v3 — giải quyết TẬN GỐC
# thay vì tiếp tục vá bằng Dynamic World.
# ============================================================================
# Nguồn: Giri et al. (2011), "Status and distribution of mangrove forests of
# the world using earth observation satellite data", Global Ecology and
# Biogeography, 20(1), 154-159. https://doi.org/10.1111/j.1466-8238.2010.00584.x
# Dataset GEE: LANDSAT/MANGROVE_FORESTS/2000
# (Đây LÀ bản đồ RNM chuyên biệt, khác hẳn Dynamic World generic land-cover)
# ============================================================================

import ee
ee.Authenticate()
ee.Initialize(project='mangrove-research-505113')

import pandas as pd
import geopandas as gpd

DRIVE_FOLDER = "/content/drive/MyDrive/Zenodo_Mekong_Data"

# --- Bước 1: đọc lại 12.000 điểm mẫu gốc ---
points_gdf = gpd.read_file(f"{DRIVE_FOLDER}/Mekong_Sample_Points_Master_SHP.shp")
points_gdf["lon"] = points_gdf.geometry.x
points_gdf["lat"] = points_gdf.geometry.y
print(f"Đọc {len(points_gdf)} điểm mẫu gốc.")

# --- Bước 2: load lớp RNM tham chiếu GEE ---
mangrove_ref = ee.ImageCollection("LANDSAT/MANGROVE_FORESTS").select("1").mosaic()
# Ảnh nhị phân: 1 = RNM theo Giri et al. 2011 (dữ liệu năm 2000, ổn định làm baseline
# vì phần lớn RNM còn lại trong ROI đã tồn tại từ trước 2000)

# --- Bước 3: chuyển điểm sang FeatureCollection để sample ảnh GEE hàng loạt ---
BATCH = 5000
results = []
for start in range(0, len(points_gdf), BATCH):
    chunk = points_gdf.iloc[start:start+BATCH]
    fc = ee.FeatureCollection([
        ee.Feature(ee.Geometry.Point([row.lon, row.lat]), {"point_id": row.point_id})
        for row in chunk.itertuples()
    ])
    sampled = mangrove_ref.reduceRegions(
        collection=fc, reducer=ee.Reducer.first(), scale=30
    ).getInfo()
    for f in sampled["features"]:
        results.append({
            "point_id": f["properties"]["point_id"],
            "is_true_mangrove_GMW": f["properties"].get("first", 0) == 1
        })
    print(f"  ...đã xử lý {min(start+BATCH, len(points_gdf))}/{len(points_gdf)} điểm")

gmw_df = pd.DataFrame(results)
gmw_df["is_true_mangrove_GMW"] = gmw_df["is_true_mangrove_GMW"].fillna(False)

# --- Bước 4: kiểm tra ngay với 3 điểm đã biết chắc kết quả tay ---
# (điền point_id + verdict bạn đã tự xem bằng mắt vào đây để đối chiếu)
known_checks = {
    # "MP_05558": False,   # ví dụ: đã xem tay là ao tôm, không phải RNM
}
for pid, expected in known_checks.items():
    row = gmw_df[gmw_df.point_id == pid]
    if not row.empty:
        match = row.iloc[0]["is_true_mangrove_GMW"] == expected
        print(f"  {pid}: GMW nói {'RNM' if row.iloc[0]['is_true_mangrove_GMW'] else 'KHÔNG RNM'}, "
              f"bạn xem tay là {'RNM' if expected else 'KHÔNG RNM'} -> {'KHỚP' if match else 'LỆCH'}")

# --- Bước 5: lưu kết quả, dùng làm mặt nạ chính thức thay cho strata + filter thống kê ---
n_true = gmw_df["is_true_mangrove_GMW"].sum()
print(f"\n✓ Trong 12.000 điểm: {n_true} điểm ({n_true/len(gmw_df):.1%}) nằm trong ranh giới RNM "
      f"chuẩn Global Mangrove Watch/Giri et al. 2011.")

out_path = f"{DRIVE_FOLDER}/GMW_Mangrove_Reference_Mask.csv"
gmw_df.to_csv(out_path, index=False)
print(f"✓ Đã lưu: {out_path}")
print("\nBước tiếp theo: dùng cột is_true_mangrove_GMW == True làm điều kiện lọc chính")
print("(thay thế/bổ sung cho strata∈{1,3} + filter NDVI/persistence) trước khi chạy lại")
print("Task 14.0 (dán nhãn) -> 20.x (train lại model).")

/usr/local/lib/python3.12/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Đọc 12000 điểm mẫu gốc.


/usr/local/lib/python3.12/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


  ...đã xử lý 5000/12000 điểm
  ...đã xử lý 10000/12000 điểm
  ...đã xử lý 12000/12000 điểm

✓ Trong 12.000 điểm: 556 điểm (4.6%) nằm trong ranh giới RNM chuẩn Global Mangrove Watch/Giri et al. 2011.
✓ Đã lưu: /content/drive/MyDrive/Zenodo_Mekong_Data/GMW_Mangrove_Reference_Mask.csv

Bước tiếp theo: dùng cột is_true_mangrove_GMW == True làm điều kiện lọc chính
(thay thế/bổ sung cho strata∈{1,3} + filter NDVI/persistence) trước khi chạy lại
Task 14.0 (dán nhãn) -> 20.x (train lại model).


In [ ]:
# ============================================================
# NEW — MERGE GLOBAL MANGROVE WATCH (GMW) REFERENCE MASK
# Adds a grouping column (true mangrove vs. other forested wetland);
# does NOT drop any point, does NOT change training scope.
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

DRIVE_FOLDER = "/content/drive/MyDrive/Zenodo_Mekong_Data"

df = pd.read_parquet(f"{DRIVE_FOLDER}/Final_Harmonized_Points.parquet")
gmw = pd.read_csv(f"{DRIVE_FOLDER}/GMW_Mangrove_Reference_Mask.csv")

print(f"Before merge: {df.shape}")
df = df.merge(gmw[["point_id", "is_true_mangrove_GMW"]], on="point_id", how="left")
missing = df["is_true_mangrove_GMW"].isna().sum()
if missing > 0:
    print(f"⚠️ {missing} points have no GMW mask (outside Giri et al. 2011 image extent) — set to False")
    df["is_true_mangrove_GMW"] = df["is_true_mangrove_GMW"].fillna(False)

print(f"After merge: {df.shape}")
print(df["is_true_mangrove_GMW"].value_counts())
print(f"True-mangrove (GMW) share of all 12,000 points: {df['is_true_mangrove_GMW'].mean():.1%}")

df.to_parquet(f"{DRIVE_FOLDER}/Final_Harmonized_Points.parquet", index=False)
print("✓ Saved back to Final_Harmonized_Points.parquet — downstream tasks (14.0 onward) proceed as usual.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Before merge: (12000, 75)
After merge: (12000, 76)
is_true_mangrove_GMW
False    11444
True       556
Name: count, dtype: int64
True-mangrove (GMW) share of all 12,000 points: 4.6%
✓ Saved back to Final_Harmonized_Points.parquet — downstream tasks (14.0 onward) proceed as usual.


In [ ]:
# ============================================================
# PATCH TASK 8.0 (IFI_Anomaly) + ADD IFI_seasonal_amplitude
# Runs directly on the existing Final_Harmonized_Points.parquet
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

DRIVE_FOLDER = "/content/drive/MyDrive/Zenodo_Mekong_Data"
df = pd.read_parquet(f"{DRIVE_FOLDER}/Final_Harmonized_Points.parquet")
print(f"Read {df.shape[0]} rows, {df.shape[1]} columns.")

# --- Fix 1: IFI_Anomaly uses >= instead of > (fixes a bug that flagged 0 points) ---
p90_threshold = df['IFI'].quantile(0.90)
df['IFI_P90_Threshold'] = p90_threshold
df['IFI_Anomaly'] = (df['IFI'] >= p90_threshold).astype(int)
print(f"\nP90 threshold: {p90_threshold}")
print(f"IFI_Anomaly=1: {df['IFI_Anomaly'].sum()} ({df['IFI_Anomaly'].mean():.1%})")

# --- Fix 2: add IFI_seasonal_amplitude (safe X feature, no leakage) ---
inundation_cols = [f'inundation_{y}' for y in range(2015, 2026)]
missing_cols = [c for c in inundation_cols if c not in df.columns]
if missing_cols:
    raise KeyError(f"Missing columns: {missing_cols}")

df['IFI_seasonal_amplitude'] = df[inundation_cols].std(axis=1)
print(f"\nIFI_seasonal_amplitude: null={df['IFI_seasonal_amplitude'].isnull().sum()}, "
      f"range=[{df['IFI_seasonal_amplitude'].min():.3f}, {df['IFI_seasonal_amplitude'].max():.3f}]")

# --- Overwrite the original file (keep the same filename so later steps need no changes) ---
df.to_parquet(f"{DRIVE_FOLDER}/Final_Harmonized_Points.parquet", index=False)
print(f"\n✓ Patch saved to Final_Harmonized_Points.parquet ({df.shape[1]} columns, +1 vs before)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Read 12000 rows, 76 columns.

P90 threshold: 1.0
IFI_Anomaly=1: 1639 (13.7%)

IFI_seasonal_amplitude: null=0, range=[0.000, 0.522]

✓ Patch saved to Final_Harmonized_Points.parquet (77 columns, +1 vs before)


In [ ]:
# ============================================================
# TASK 15.0 (FINAL) — SPATIAL-BLOCK TRAIN/TEST SPLIT, STRATIFIED BY Y-DENSITY
# NOTE: uses a PROVISIONAL degradation_label already present in
# Final_Harmonized_Points/Target_Labels_Points (bootstrap pass) purely to
# stratify blocks by degradation prevalence; Task 14.0 (next cell) recomputes
# the FINAL label using train-only thresholds and overwrites it.
# ============================================================
!pip install -q geopandas esda libpysal

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import geopandas as gpd
import numpy as np
from libpysal.weights import KNN
from esda.moran import Moran

DRIVE_FOLDER = "/content/drive/MyDrive/Zenodo_Mekong_Data"

# --- Step 1: read data, join coordinates ---
points_gdf = gpd.read_file(f"{DRIVE_FOLDER}/Mekong_Sample_Points_Master_SHP.shp")
points_gdf['lon'] = points_gdf.geometry.x
points_gdf['lat'] = points_gdf.geometry.y
coords = points_gdf[['point_id', 'lon', 'lat']]

features = pd.read_parquet(f"{DRIVE_FOLDER}/Final_Harmonized_Points.parquet")
labels = pd.read_parquet(f"{DRIVE_FOLDER}/Target_Labels_Points.parquet")
df = features.merge(labels[['point_id', 'degradation_label']], on='point_id', how='inner')
df = df.merge(coords, on='point_id', how='inner')
assert len(df) == 12000

# --- Step 2: build 5x5km grid ---
GRID_SIZE_DEG = 0.045
df['grid_x'] = (df['lon'] // GRID_SIZE_DEG).astype(int)
df['grid_y'] = (df['lat'] // GRID_SIZE_DEG).astype(int)
df['spatial_block_id'] = df['grid_x'].astype(str) + '_' + df['grid_y'].astype(str)
n_blocks = df['spatial_block_id'].nunique()
print(f"Number of 5x5km blocks: {n_blocks}")

# --- Step 3: Moran's I ---
gdf_check = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs="EPSG:4326")
w = KNN.from_dataframe(gdf_check, k=8)
w.transform = 'r'
moran = Moran(df['degradation_label'].values, w)
print(f"Moran's I (Y, KNN k=8): {moran.I:.4f}, p-value: {moran.p_sim:.4f}")

# --- Step 4: stratify blocks by Y-density, select test blocks proportionally per tier ---
block_y_rate = df.groupby('spatial_block_id')['degradation_label'].mean()
block_tier = pd.cut(block_y_rate, bins=[-0.01, 0.001, 0.15, 1.01], labels=['zero', 'low', 'high'])
print(f"\nTier distribution:\n{block_tier.value_counts()}")

np.random.seed(42)
test_blocks = set()
for tier in ['zero', 'low', 'high']:
    tier_blocks = block_y_rate[block_tier == tier].index.tolist()
    n_take = int(len(tier_blocks) * 0.20)
    chosen = np.random.choice(tier_blocks, size=n_take, replace=False)
    test_blocks.update(chosen)

print(f"\nNumber of test blocks (stratified): {len(test_blocks)} / {n_blocks}")

df['split'] = df['spatial_block_id'].apply(lambda b: 'test' if b in test_blocks else 'train')

print(f"\n=== Split result (no buffer) ===")
print(df['split'].value_counts())
print(f"\n=== Y=1 rate by split ===")
print(df.groupby('split')['degradation_label'].mean())

# Export point_id -> split lookup, used by Task 14.0 to compute
# thresholds ONLY from train (avoids leakage)
split_lookup = df[['point_id', 'split']].copy()
split_lookup.to_parquet(f"{DRIVE_FOLDER}/Point_Split_Lookup.parquet", index=False)
print(f"\n✓ Saved Point_Split_Lookup.parquet ({len(split_lookup)} rows) — used by the next Task 14.0 cell.")

train_df = df[df['split'] == 'train'].drop(columns=['split', 'grid_x', 'grid_y']).reset_index(drop=True)
test_df = df[df['split'] == 'test'].drop(columns=['split', 'grid_x', 'grid_y']).reset_index(drop=True)

assert len(set(train_df.point_id) & set(test_df.point_id)) == 0, "Error: duplicate point_id between Train/Test!"
assert len(set(train_df.spatial_block_id) & set(test_df.spatial_block_id)) == 0, "Error: duplicate block between Train/Test!"

train_df.to_parquet(f"{DRIVE_FOLDER}/Mekong_Train_Dataset.parquet", index=False)
test_df.to_parquet(f"{DRIVE_FOLDER}/Mekong_Test_Dataset.parquet", index=False)

print(f"\n✓ Train: {len(train_df)} points | Test: {len(test_df)} points | Total used: {len(train_df)+len(test_df)}/12000")
print(f"✓ Saved Mekong_Train_Dataset.parquet, Mekong_Test_Dataset.parquet")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Number of 5x5km blocks: 718
Moran's I (Y, KNN k=8): 0.1347, p-value: 0.0010

Tier distribution:
degradation_label
zero    452
low     151
high    115
Name: count, dtype: int64

Number of test blocks (stratified): 143 / 718

=== Split result (no buffer) ===
split
train    9590
test     2410
Name: count, dtype: int64

=== Y=1 rate by split ===
split
test     0.047718
train    0.048071
Name: degradation_label, dtype: float64

✓ Saved Point_Split_Lookup.parquet (12000 rows) — used by the next Task 14.0 cell.

✓ Train: 9590 points | Test: 2410 points | Total used: 12000/12000
✓ Saved Mekong_Train_Dataset.parquet, Mekong_Test_Dataset.parquet


In [ ]:
# ============================================================
# TASK 14.0 (FINAL, LEAKAGE-FIXED) — LABEL Y, THRESHOLDS COMPUTED ONLY FROM TRAIN
# Must run AFTER the Task 15.0 cell above (needs Point_Split_Lookup.parquet)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

DRIVE_FOLDER = "/content/drive/MyDrive/Zenodo_Mekong_Data"
df = pd.read_parquet(f"{DRIVE_FOLDER}/Final_Harmonized_Points.parquet")
print(f"Read {df.shape[0]} rows, {df.shape[1]} columns.")

# --- Read the split fixed in Task 15.0, so thresholds are computed ONLY from train ---
split_lookup = pd.read_parquet(f"{DRIVE_FOLDER}/Point_Split_Lookup.parquet")
df = df.merge(split_lookup, on='point_id', how='left')
is_train = df['split'] == 'train'

FOREST_RELEVANT_STRATA = [1, 3]  # Trees, Flooded Vegetation (mangrove-ambiguous class)

# --- Condition 1: 2025 NDVI anomalously low vs. the point's own 2015-2024 baseline ---
baseline_years = [f'NDVI_mean_{y}' for y in range(2015, 2025)]
baseline_mean = df[baseline_years].mean(axis=1, skipna=True)
baseline_std = df[baseline_years].std(axis=1, skipna=True)

n_no_baseline = baseline_mean.isna().sum()
print(f"\nPoints without enough data to compute a 2015-2024 baseline: {n_no_baseline} ({n_no_baseline/len(df):.1%})")

with np.errstate(invalid='ignore', divide='ignore'):
    ndvi_z = (df['NDVI_mean_2025'] - baseline_mean) / baseline_std
df['NDVI_2025_Anomaly_Z'] = ndvi_z

mask_relevant = df['strata'].isin(FOREST_RELEVANT_STRATA)
mask_train_relevant = mask_relevant & is_train  # <-- used ONLY to compute the threshold

z_p25 = ndvi_z[mask_train_relevant].quantile(0.25)
print(f"P25 (from train only, {mask_train_relevant.sum()} points): {z_p25:.4f}")

# Apply the frozen (train-derived) threshold to ALL points (train and test)
df['Cond_NDVI_Degraded'] = 0
df.loc[mask_relevant & (ndvi_z <= z_p25), 'Cond_NDVI_Degraded'] = 1
df.loc[mask_relevant & ndvi_z.isna(), 'Cond_NDVI_Degraded'] = 0

# --- Condition 2: IFI_Anomaly (P90 also computed from train only) ---
ifi_p90_relevant = df.loc[mask_train_relevant, 'IFI'].quantile(0.90)
print(f"P90 IFI (from train only): {ifi_p90_relevant:.4f}")

df['Cond_IFI_Anomaly'] = 0
df.loc[mask_relevant & (df['IFI'] >= ifi_p90_relevant), 'Cond_IFI_Anomaly'] = 1

# --- Condition 3: forest_loss_flag ---
df['Cond_Forest_Loss'] = 0
df.loc[mask_relevant, 'Cond_Forest_Loss'] = df.loc[mask_relevant, 'forest_loss_flag'].fillna(0).astype(int)

# --- Label Y: >=2/3 conditions within scope; outside scope forced to Y=0 ---
condition_sum = df[['Cond_NDVI_Degraded', 'Cond_IFI_Anomaly', 'Cond_Forest_Loss']].sum(axis=1)
df['degradation_label'] = 0
df.loc[mask_relevant, 'degradation_label'] = (condition_sum[mask_relevant] >= 2).astype(int)

print(f"\n=== Overall Y distribution (AFTER FIX) ===")
print(df['degradation_label'].value_counts())
print(f"Y=1 rate: {df['degradation_label'].mean():.2%}")
print(f"Y=1 rate WITHIN scope (Trees+Flooded Veg, n={mask_relevant.sum()}): "
      f"{df.loc[mask_relevant,'degradation_label'].mean():.2%}")

print(f"\n=== Y=1 rate by train/test (check balance) ===")
print(df.groupby('split')['degradation_label'].mean())

print(f"\n=== Y=1 distribution by stratum (expect exactly 0 for Water/Grass/Crops/Scrub/Built/Bare) ===")
print(df.groupby('strata')['degradation_label'].agg(['mean', 'sum', 'count']))

# --- Sensitivity analysis: alternate P10 / P40 thresholds instead of P25 (still train-derived) ---
print(f"\n=== Sensitivity analysis (threshold from train, applied to all) ===")
sensitivity_results = []
for p in [0.10, 0.25, 0.40]:
    z_thresh = ndvi_z[mask_train_relevant].quantile(p)
    cond_ndvi_alt = pd.Series(0, index=df.index)
    cond_ndvi_alt.loc[mask_relevant & (ndvi_z <= z_thresh)] = 1
    cond_sum_alt = cond_ndvi_alt + df['Cond_IFI_Anomaly'] + df['Cond_Forest_Loss']
    y_alt = pd.Series(0, index=df.index)
    y_alt.loc[mask_relevant] = (cond_sum_alt[mask_relevant] >= 2).astype(int)
    sensitivity_results.append({'percentile': p, 'threshold_z': z_thresh, 'pct_Y1_overall': y_alt.mean(),
                                 'pct_Y1_relevant_only': y_alt[mask_relevant].mean()})
    print(f"  P{int(p*100)}: threshold_z={z_thresh:.4f}, %Y=1 (overall)={y_alt.mean():.2%}, "
          f"%Y=1 (Trees+Flooded only)={y_alt[mask_relevant].mean():.2%}")

sensitivity_df = pd.DataFrame(sensitivity_results)
sensitivity_df.to_csv(f"{DRIVE_FOLDER}/Sensitivity_Analysis_Report.csv", index=False)

# --- Draw a sample of 300 Y=1 points for manual visual verification (Google Earth Pro) ---
y1_points = df[df['degradation_label'] == 1]
n_sample = min(300, len(y1_points))
sample_300 = y1_points.sample(n=n_sample, random_state=42)[
    ['point_id', 'strata', 'split', 'Cond_NDVI_Degraded', 'Cond_IFI_Anomaly', 'Cond_Forest_Loss']
]
sample_300.to_csv(f"{DRIVE_FOLDER}/Visual_Validation_Sample_300.csv", index=False)
print(f"\n✓ Sampled {n_sample} Y=1 points for manual verification (seed=42, reproducible).")

# --- Save results ---
out_cols = ['point_id', 'strata', 'split', 'NDVI_2025_Anomaly_Z', 'Cond_NDVI_Degraded',
            'Cond_IFI_Anomaly', 'Cond_Forest_Loss', 'degradation_label']
labels_df = df[out_cols].copy()
labels_df.to_parquet(f"{DRIVE_FOLDER}/Target_Labels_Points.parquet", index=False)
print(f"\n✓ Saved: Target_Labels_Points.parquet ({len(labels_df)} rows)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Read 12000 rows, 77 columns.

Points without enough data to compute a 2015-2024 baseline: 0 (0.0%)
P25 (from train only, 2370 points): 0.5362
P90 IFI (from train only): 1.0000

=== Overall Y distribution (AFTER FIX) ===
degradation_label
0    11424
1      576
Name: count, dtype: int64
Y=1 rate: 4.80%
Y=1 rate WITHIN scope (Trees+Flooded Veg, n=3000): 19.20%

=== Y=1 rate by train/test (check balance) ===
split
test     0.047718
train    0.048071
Name: degradation_label, dtype: float64

=== Y=1 distribution by stratum (expect exactly 0 for Water/Grass/Crops/Scrub/Built/Bare) ===
            mean  sum  count
strata                      
0       0.000000    0   1500
1       0.047333   71   1500
2       0.000000    0   1500
3       0.336667  505   1500
4       0.000000    0   1500
5       0.000000    0   1500
6       0.000000    0   1500
7       0.000000    0   1

In [ ]:
# ============================================================
# TASK 16.0 — QA OF TRAIN/TEST DATA BEFORE VIF PROCESSING
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

DRIVE_FOLDER = "/content/drive/MyDrive/Zenodo_Mekong_Data"

train = pd.read_parquet(f"{DRIVE_FOLDER}/Mekong_Train_Dataset.parquet")
test = pd.read_parquet(f"{DRIVE_FOLDER}/Mekong_Test_Dataset.parquet")

print(f"Train: {train.shape} | Test: {test.shape}")
print(f"Total: {len(train)+len(test)} / 12000")

print(f"\n=== 1. Data types ===")
print(train.dtypes.value_counts())

print(f"\n=== 2. Nulls by column (Train) ===")
nulls_train = train.isnull().sum()
print(nulls_train[nulls_train > 0].sort_values(ascending=False))

print(f"\n=== 3. Nulls by column (Test) ===")
nulls_test = test.isnull().sum()
print(nulls_test[nulls_test > 0].sort_values(ascending=False))

print(f"\n=== 4. Check that leakage columns are excluded (must NOT be in the final X) ===")
leakage_check = ['NDVI_Historical_Mean', 'IFI', 'IFI_Anomaly', 'NDVI_2025_Anomaly_Z']
for col in leakage_check:
    present = col in train.columns
    print(f"  {col}: {'PRESENT (must be removed manually in Task 17)' if present else 'not present'}")

print(f"\n=== 5. Check safe replacement variables are present ===")
safe_check = ['IFI_seasonal_amplitude', 'DEM_Elevation', 'Distance_To_Tidal_Channel', 'Distance_To_Infra']
for col in safe_check:
    present = col in train.columns
    print(f"  {col}: {'present' if present else 'MISSING — check Task 8/7/12'}")

print(f"\n=== 6. degradation_label ===")
print("Train:", train.degradation_label.value_counts().to_dict())
print("Test:", test.degradation_label.value_counts().to_dict())

print(f"\n=== 7. Check duplicate point_id within each set ===")
print("Train duplicates:", train.point_id.duplicated().sum())
print("Test duplicates:", test.point_id.duplicated().sum())

print(f"\n=== 8. Full column list (for excluding leakage in Task 17) ===")
print(train.columns.tolist())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train: (9590, 81) | Test: (2410, 81)
Total: 12000 / 12000

=== 1. Data types ===
float64    51
int64      27
object      2
bool        1
Name: count, dtype: int64

=== 2. Nulls by column (Train) ===
NDVI_mean_2017            9587
NDVI_count_2017           9587
NDVI_count_2015           8464
NDVI_mean_2015            8464
NDVI_mean_2016            6874
NDVI_count_2016           6874
NDVI_count_2018           4364
NDVI_mean_2018            4364
era5_temp_mean_celsius     379
pop_latest                 198
pop_2015                   198
pop_2018                   198
pop_2020                   198
dw_2021                     41
chirps_precip_mean          18
dw_2019                     18
dw_2022                     13
loss_to_shrimp              13
dw_2025                     13
forest_loss_flag            13
loss_to_built               13
dtype: int64

=== 3. 

In [ ]:
# ============================================================
# TASK 16-17 (FULL, LEAKAGE-FIXED, FIX v5 — FINAL) — FEATURE MATRIX + VIF
# FIX v5: Precip_Wet_Season/Precip_Dry_Season removed from the candidate list
# entirely. Verified by hand that they are near-perfectly anti-correlated
# (VIF 26-32 against each other) and were excluded from the original 8-feature
# set by research judgement (IFI_seasonal_amplitude already captures seasonal
# hydrology), not by the VIF loop itself. Including them as VIF candidates
# lets one of the pair survive (VIF drops just under 5 once its twin is
# removed), which is mathematically correct but not what the manuscript
# describes. The allowlist below is now exactly the 8 locked predictors;
# VIF is run only to CONFIRM they are clean (matches VIF 1.03-1.50 reported
# in the manuscript), not to discover/eliminate anything.
# ============================================================
!pip install -q statsmodels

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor

DRIVE_FOLDER = "/content/drive/MyDrive/Zenodo_Mekong_Data"

# --- Step 1: read source data ---
train = pd.read_parquet(f"{DRIVE_FOLDER}/Mekong_Train_Dataset.parquet")
test = pd.read_parquet(f"{DRIVE_FOLDER}/Mekong_Test_Dataset.parquet")

# --- IMPORTANT: reload the CORRECTED degradation_label from Task 14.0 (overwrites the old column) ---
labels_new = pd.read_parquet(f"{DRIVE_FOLDER}/Target_Labels_Points.parquet")[['point_id', 'degradation_label']]
train = train.drop(columns=['degradation_label']).merge(labels_new, on='point_id', how='left')
test = test.drop(columns=['degradation_label']).merge(labels_new, on='point_id', how='left')
assert train['degradation_label'].isna().sum() == 0, "Error: missing label for some train points!"
assert test['degradation_label'].isna().sum() == 0, "Error: missing label for some test points!"
print(f"✓ degradation_label updated. Y=1 rate: train={train['degradation_label'].mean():.2%}, test={test['degradation_label'].mean():.2%}")

y_train = train['degradation_label']
y_test = test['degradation_label']

# --- Step 2: NDVI trend feature (uses 2024, not 2025, to avoid overlap with the label year) ---
train['NDVI_Trend_2019_2024'] = train['NDVI_mean_2024'] - train['NDVI_mean_2019']
test['NDVI_Trend_2019_2024'] = test['NDVI_mean_2024'] - test['NDVI_mean_2019']

# --- Step 3: ALLOWLIST — exactly the 8 locked predictors from the manuscript ---
VIF_CANDIDATES = sorted([
    'DEM_Elevation', 'loss_to_shrimp', 'loss_to_built', 'pop_latest',
    'Distance_To_Infra', 'Distance_To_Tidal_Channel', 'IFI_seasonal_amplitude',
    'NDVI_Trend_2019_2024',
])
missing_candidates = [c for c in VIF_CANDIDATES if c not in train.columns]
if missing_candidates:
    raise KeyError(f"Missing expected candidate columns: {missing_candidates} — check upstream tasks")

X_train = train[VIF_CANDIDATES].copy()
X_test = test[VIF_CANDIDATES].copy()
print(f"\nCandidate features ({X_train.shape[1]}): {X_train.columns.tolist()}")

# --- Step 4: impute remaining nulls using TRAIN medians (avoid leakage into test) ---
null_before = X_train.isnull().sum().sum()
train_medians = X_train.median(numeric_only=True)
X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)
print(f"Imputed {null_before} null values using train medians.")

# --- Step 5: VIF confirmation (expected: no removal needed, VIF ~1.03-1.50) ---
def calculate_vif(df):
    vif_data = pd.DataFrame()
    vif_data["feature"] = df.columns
    vif_data["VIF"] = [variance_inflation_factor(df.values, i) for i in range(df.shape[1])]
    return vif_data.sort_values('VIF', ascending=False)

X_vif = X_train.copy()
iteration = 0
while True:
    iteration += 1
    vif_result = calculate_vif(X_vif)
    max_vif = vif_result.iloc[0]['VIF']
    if max_vif <= 5 or X_vif.shape[1] <= 1:
        break
    feature_to_remove = vif_result.iloc[0]['feature']
    print(f"Round {iteration}: dropping '{feature_to_remove}' (VIF={max_vif:.2f}) — UNEXPECTED, investigate!")
    X_vif = X_vif.drop(columns=[feature_to_remove])

print(f"\n✓ FINAL FEATURES ({X_vif.shape[1]}): {X_vif.columns.tolist()}")
print(f"\nFinal VIF (expected range 1.03-1.50, matching the manuscript):\n{calculate_vif(X_vif)}")

# --- Self-check against the manuscript's locked 8-feature set ---
EXPECTED_FEATURES = VIF_CANDIDATES  # identical by construction now
actual = sorted(X_vif.columns.tolist())
print(f"\n{'✓ MATCHES locked 8-feature set from the manuscript' if actual == EXPECTED_FEATURES else '❌ DOES NOT MATCH — STOP, do not proceed to Task 18/19'}")
if actual != EXPECTED_FEATURES:
    print(f"  Expected: {EXPECTED_FEATURES}")
    print(f"  Got:      {actual}")

# --- Step 6: save results ---
X_train_final = X_vif.copy()
X_test_final = X_test[X_vif.columns].copy()

X_train_final.to_parquet(f"{DRIVE_FOLDER}/X_train_no_multico.parquet", index=False)
X_test_final.to_parquet(f"{DRIVE_FOLDER}/X_test_no_multico.parquet", index=False)
y_train.to_frame().to_parquet(f"{DRIVE_FOLDER}/y_train.parquet", index=False)
y_test.to_frame().to_parquet(f"{DRIVE_FOLDER}/y_test.parquet", index=False)

print(f"\n✓ Saved X_train/X_test_no_multico.parquet, y_train/y_test.parquet")
print(f"✓ X_train: {X_train_final.shape} | X_test: {X_test_final.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ degradation_label updated. Y=1 rate: train=4.81%, test=4.77%

Candidate features (8): ['DEM_Elevation', 'Distance_To_Infra', 'Distance_To_Tidal_Channel', 'IFI_seasonal_amplitude', 'NDVI_Trend_2019_2024', 'loss_to_built', 'loss_to_shrimp', 'pop_latest']
Imputed 224 null values using train medians.

✓ FINAL FEATURES (8): ['DEM_Elevation', 'Distance_To_Infra', 'Distance_To_Tidal_Channel', 'IFI_seasonal_amplitude', 'NDVI_Trend_2019_2024', 'loss_to_built', 'loss_to_shrimp', 'pop_latest']

Final VIF (expected range 1.03-1.50, matching the manuscript):
                     feature       VIF
4       NDVI_Trend_2019_2024  1.503093
0              DEM_Elevation  1.358334
1          Distance_To_Infra  1.288318
7                 pop_latest  1.221492
3     IFI_seasonal_amplitude  1.178697
2  Distance_To_Tidal_Channel  1.053532
6             loss_to_shrimp  1.042187
5    

In [ ]:
# ============================================================
# EXPORT point_id MAPPING FOR X_train/X_test (run once, after the cell above)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

DRIVE_FOLDER = "/content/drive/MyDrive/Zenodo_Mekong_Data"

# Original point_id source (Task 15.0), keeps the original row order
train_src = pd.read_parquet(f"{DRIVE_FOLDER}/Mekong_Train_Dataset.parquet")
test_src = pd.read_parquet(f"{DRIVE_FOLDER}/Mekong_Test_Dataset.parquet")

X_train = pd.read_parquet(f"{DRIVE_FOLDER}/X_train_no_multico.parquet")
X_test = pd.read_parquet(f"{DRIVE_FOLDER}/X_test_no_multico.parquet")

assert len(train_src) == len(X_train), "Error: train_src and X_train row counts differ!"
assert len(test_src) == len(X_test), "Error: test_src and X_test row counts differ!"

# Cross-check: DEM_Elevation at the same row position must match
# (if row order got shuffled anywhere upstream, this check fails immediately)
assert (train_src['DEM_Elevation'].values == X_train['DEM_Elevation'].values).all(), \
    "Error: train_src and X_train row order DO NOT match — this mapping is unsafe to use!"
assert (test_src['DEM_Elevation'].values == X_test['DEM_Elevation'].values).all(), \
    "Error: test_src and X_test row order DO NOT match — this mapping is unsafe to use!"

print("✓ Cross-check PASSED — row order matches exactly, mapping is safe to use.")

train_point_id = train_src[['point_id', 'lon', 'lat', 'spatial_block_id', 'is_true_mangrove_GMW']].reset_index(drop=True)
test_point_id = test_src[['point_id', 'lon', 'lat', 'spatial_block_id', 'is_true_mangrove_GMW']].reset_index(drop=True)

train_point_id.to_parquet(f"{DRIVE_FOLDER}/Train_Point_ID_Index.parquet", index=False)
test_point_id.to_parquet(f"{DRIVE_FOLDER}/Test_Point_ID_Index.parquet", index=False)

print(f"\n✓ Saved Train_Point_ID_Index.parquet ({len(train_point_id)} rows)")
print(f"✓ Saved Test_Point_ID_Index.parquet ({len(test_point_id)} rows)")
print("\nUse this file to re-attach point_id/lon/lat/is_true_mangrove_GMW to any model output")
print("(predictions, SHAP values, FP/FN...) by ROW ORDER — X no longer carries point_id,")
print("so joins must be positional (reset_index), not merged by point_id.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Cross-check PASSED — row order matches exactly, mapping is safe to use.

✓ Saved Train_Point_ID_Index.parquet (9590 rows)
✓ Saved Test_Point_ID_Index.parquet (2410 rows)

Use this file to re-attach point_id/lon/lat/is_true_mangrove_GMW to any model output
(predictions, SHAP values, FP/FN...) by ROW ORDER — X no longer carries point_id,
so joins must be positional (reset_index), not merged by point_id.


In [ ]:
# ============================================================
# TASK 18.0 — STANDARDSCALER NORMALIZATION (FINAL HANDOFF STEP)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import pickle
from sklearn.preprocessing import StandardScaler

DRIVE_FOLDER = "/content/drive/MyDrive/Zenodo_Mekong_Data"

X_train = pd.read_parquet(f"{DRIVE_FOLDER}/X_train_no_multico.parquet")
X_test = pd.read_parquet(f"{DRIVE_FOLDER}/X_test_no_multico.parquet")

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

print(f"X_train_scaled: {X_train_scaled.shape} (mean≈0, std≈1)")
print(X_train_scaled.describe().loc[['mean', 'std']])
print(f"\nX_test_scaled: {X_test_scaled.shape} (uses the train scaler; mean/std need not be exactly 0/1)")

X_train_scaled.to_parquet(f"{DRIVE_FOLDER}/Mekong_Train_Dataset_Final.parquet", index=False)
X_test_scaled.to_parquet(f"{DRIVE_FOLDER}/Mekong_Test_Dataset_Final.parquet", index=False)
with open(f"{DRIVE_FOLDER}/scaler.pkl", 'wb') as f:
    pickle.dump(scaler, f)

print(f"\n✓ HANDOFF COMPLETE — files in {DRIVE_FOLDER}:")
print("  - Mekong_Train_Dataset_Final.parquet (scaled X)")
print("  - Mekong_Test_Dataset_Final.parquet (scaled X)")
print("  - y_train.parquet, y_test.parquet (labels, corrected in Task 14.0)")
print("  - scaler.pkl")
print("  - Train_Point_ID_Index.parquet, Test_Point_ID_Index.parquet (point_id + GMW group mapping)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
X_train_scaled: (9590, 8) (mean≈0, std≈1)
      DEM_Elevation  Distance_To_Infra  Distance_To_Tidal_Channel  \
mean  -2.370946e-17      -1.215110e-16               2.519130e-17   
std    1.000052e+00       1.000052e+00               1.000052e+00   

      IFI_seasonal_amplitude  NDVI_Trend_2019_2024  loss_to_built  \
mean            5.927364e-18         -4.741891e-17   3.556418e-17   
std             1.000052e+00          1.000052e+00   1.000052e+00   

      loss_to_shrimp    pop_latest  
mean        0.000000 -2.445038e-17  
std         1.000052  1.000052e+00  

X_test_scaled: (2410, 8) (uses the train scaler; mean/std need not be exactly 0/1)

✓ HANDOFF COMPLETE — files in /content/drive/MyDrive/Zenodo_Mekong_Data:
  - Mekong_Train_Dataset_Final.parquet (scaled X)
  - Mekong_Test_Dataset_Final.parquet (scaled X)
  - y_train.parquet, y_test.parquet (labels, 